<a href="https://colab.research.google.com/github/hamza-24-ai/ca_research/blob/main/CA_TASK_03_CNN_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Running CNN model (Image Net V1)

## CPU Model Test

In [1]:
import torch, time, statistics, csv, torchvision
from torch.utils.flop_counter import FlopCounterMode

DEVICE = "cpu"          # change to "cuda" (T4) or "tpu" later
if DEVICE == "tpu":
    import torch_tpu
dev = torch.device(DEVICE)
print("PyTorch", torch.__version__, "| device:", dev)

def sync(out):
    # forces the device to finish before the timer stops
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "tpu":
        out.sum().item()

# ---- model ----
model = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval().to(dev)
params = sum(p.numel() for p in model.parameters())

# ---- FLOPs (per image, counted by PyTorch FlopCounterMode) ----
with FlopCounterMode(display=False) as fc:
    model(torch.randn(1, 3, 224, 224, device=dev))
flops_per_img = fc.get_total_flops()
print("Params:", params, "| FLOPs/img:", flops_per_img)

# ---- startup time (first run, includes setup/compile) ----
t0 = time.perf_counter()
with torch.inference_mode():
    out = model(torch.randn(1, 3, 224, 224, device=dev))
    sync(out)
startup_s = time.perf_counter() - t0
print("Startup (first run):", round(startup_s, 4), "s")

def bench(bs, dtype_name, dtype, reps):
    m = model.to(dtype)
    x = torch.randn(bs, 3, 224, 224, device=dev, dtype=dtype)
    with torch.inference_mode():
        for _ in range(3):                      # warm-up, not timed
            sync(m(x))
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter()
            out = m(x)
            sync(out)                           # includes forward pass only
            ts.append(time.perf_counter() - t0)
    med = statistics.median(ts)
    ts_sorted = sorted(ts)
    p95 = ts_sorted[int(0.95 * (len(ts) - 1))]
    p99 = ts_sorted[int(0.99 * (len(ts) - 1))]
    return {"device": DEVICE, "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "mean_s": statistics.mean(ts),
            "std_s": statistics.stdev(ts), "p95_s": p95, "p99_s": p99,
            "throughput_img_s": bs / med,
            "gflops_per_s": flops_per_img * bs / med / 1e9, "status": "ok"}

rows = []
for dtype_name, dtype in [("fp32", torch.float32)]:   # add ("bf16", torch.bfloat16) next
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            reps = 100 if bs == 1 else 10   # 100 runs so p95/p99 are meaningful at bs=1
            rows.append(bench(bs, dtype_name, dtype, reps))
        except Exception as e:
            rows.append({"device": DEVICE, "precision": dtype_name, "batch": bs,
                         "status": f"failed: {type(e).__name__}"})
        print(rows[-1])

with open(f"mobilenetv2_inference_{DEVICE}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved CSV")

PyTorch 2.14.1+cpu | device: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 157MB/s]


Params: 3504872 | FLOPs/img: 601548544
Startup (first run): 0.034 s
{'device': 'cpu', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.03272050399999671, 'mean_s': 0.033668963529993906, 'std_s': 0.0028714886170020985, 'p95_s': 0.038775257999986934, 'p99_s': 0.04287073599999758, 'throughput_img_s': 30.56187643075732, 'gflops_per_s': 18.384452268829982, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.06501754749999122, 'mean_s': 0.06536785910000731, 'std_s': 0.005234660843101265, 'p95_s': 0.0678250320000302, 'p99_s': 0.0678250320000302, 'throughput_img_s': 30.76092650219189, 'gflops_per_s': 18.504190549484544, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.175476682999971, 'mean_s': 0.1795230105999906, 'std_s': 0.016286151252502712, 'p95_s': 0.18615731599999208, 'p99_s': 0.18615731599999208, 'throughput_img_s': 22.79505135164118, 'gflops_per_s': 13.712329950984985, 'status': 'ok'}
{'devi

## Correct StartUp Time

In [2]:
import time, torch, torchvision
dev = torch.device("cpu")
t0 = time.perf_counter()
model = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval().to(dev)
t_load = time.perf_counter() - t0
x = torch.randn(1, 3, 224, 224, device=dev)
t1 = time.perf_counter()
with torch.inference_mode():
    model(x)
t_first = time.perf_counter() - t1
print("load_s:", round(t_load, 3), "| first_forward_s:", round(t_first, 4))

load_s: 0.113 | first_forward_s: 0.0357


### DownLoad Data Set

In [1]:
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
!tar -xzf imagenette2-320.tgz
!ls imagenette2-320/val

n01440764  n02979186  n03028079  n03417042  n03445777
n02102040  n03000684  n03394916  n03425413  n03888257


## CPU Speed + Accuracy

In [4]:
import copy, time, statistics, csv, torch, torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
dev = torch.device("cpu")
base = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval()

# ---------- speed (FP32 and BF16), forward pass only ----------
def bench(m, dtype_name, dtype, bs, reps):
    x = torch.randn(bs, 3, 224, 224, dtype=dtype)
    with torch.inference_mode():
        for _ in range(3):
            m(x)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(x); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cpu", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_img_s": bs / med, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}"})
        print(rows[-1])

# ---------- accuracy on the same held-out subset ----------
tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                         transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
ds = datasets.ImageFolder("imagenette2-320/val", transform=tf)
g = torch.Generator().manual_seed(0)
idx = torch.randperm(len(ds), generator=g)[:1000].tolist()   # fixed 1000 images
loader = DataLoader(Subset(ds, idx), batch_size=32, num_workers=0)

# Imagenette folder order -> ImageNet-1k class index (verify once against the class list)
IMAGENET_IDX = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])

def accuracy(m, dtype):
    correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            logits = m(x.to(dtype))[:, IMAGENET_IDX]     # keep the 10 relevant classes
            correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return correct / total

for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    acc = accuracy(m, dt)
    rows.append({"device": "cpu", "precision": name, "batch": "accuracy",
                 "reps": 1000, "accuracy": acc, "status": "ok"})
    print(name, "accuracy:", acc)

with open("mobilenetv2_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved mobilenetv2_cpu.csv")

{'device': 'cpu', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.029043485499983035, 'std_s': 0.03955707929011986, 'p95_s': 0.159072894000019, 'p99_s': 0.18764908899993316, 'throughput_img_s': 34.43112914256053, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.054991644499978065, 'std_s': 0.00719639108442325, 'p95_s': 0.06058139200013102, 'p99_s': 0.06058139200013102, 'throughput_img_s': 36.36916150054028, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.14167275049999262, 'std_s': 0.008729710884366833, 'p95_s': 0.14540202199987107, 'p99_s': 0.14540202199987107, 'throughput_img_s': 28.234081613317787, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 8, 'reps': 10, 'median_s': 0.34265115950006475, 'std_s': 0.028540589037336892, 'p95_s': 0.3774154699999599, 'p99_s': 0.3774154699999599, 'throughput_img_s': 23.347360072185857, 'status': 'ok'}
{'device': 'cpu', 'precision': 'f

## GPU Model Testing

In [3]:
import copy, time, statistics, csv, torch, torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
dev = torch.device("cuda")
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

def sync():
    torch.cuda.synchronize()

# ---------- startup (model load, then first forward incl. CUDA init) ----------
t0 = time.perf_counter()
base = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval()
t_load = time.perf_counter() - t0
x1 = torch.randn(1, 3, 224, 224, device=dev)
base = base.to(dev)
t1 = time.perf_counter()
with torch.inference_mode():
    base(x1)
sync()
t_first = time.perf_counter() - t1
print("load_s:", round(t_load, 3), "| first_forward_s (incl. CUDA init):", round(t_first, 4))

# ---------- speed ----------
def bench(m, dtype_name, dtype, bs, reps):
    x = torch.randn(bs, 3, 224, 224, device=dev, dtype=dtype)
    with torch.inference_mode():
        for _ in range(3):
            m(x)
        sync()
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(x); sync(); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cuda", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_img_s": bs / med, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cuda", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}"})
        print(rows[-1])

# ---------- accuracy: same 1000-image subset as CPU (seed 0) ----------
tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                         transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
ds = datasets.ImageFolder("imagenette2-320/val", transform=tf)
g = torch.Generator().manual_seed(0)
idx = torch.randperm(len(ds), generator=g)[:1000].tolist()
loader = DataLoader(Subset(ds, idx), batch_size=32, num_workers=0)

IMAGENET_IDX = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])

def accuracy(m, dtype):
    correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            logits = m(x.to(dev, dtype))[:, IMAGENET_IDX.to(dev)]
            correct += (logits.argmax(1).cpu() == y).sum().item(); total += len(y)
    return correct / total

for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    acc = accuracy(m, dt)
    rows.append({"device": "cuda", "precision": name, "batch": "accuracy", "reps": 1000,
                 "accuracy": acc, "status": "ok"})
    print(name, "accuracy:", acc)

# ---------- class mapping check (verify once) ----------
cats = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1.meta["categories"]
folders = ds.classes
for f, i in zip(folders, IMAGENET_IDX.tolist()):
    print(f, "->", i, cats[i])

with open("mobilenetv2_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved mobilenetv2_gpu.csv")

PyTorch 2.14.1+cu130 | GPU: Tesla T4
load_s: 0.092 | first_forward_s (incl. CUDA init): 0.0061
{'device': 'cuda', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.005284438000018099, 'std_s': 0.0010488254378308276, 'p95_s': 0.007631155000012768, 'p99_s': 0.010083982000026026, 'throughput_img_s': 189.23488174079722, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.0051208794999979546, 'std_s': 0.00024050268529388782, 'p95_s': 0.005456382000033955, 'p99_s': 0.005456382000033955, 'throughput_img_s': 390.55791099962397, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.005533939999992299, 'std_s': 2.2378576560720707e-05, 'p95_s': 0.0055769339999756085, 'p99_s': 0.0055769339999756085, 'throughput_img_s': 722.8123181685321, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 8, 'reps': 10, 'median_s': 0.009597357499984582, 'std_s': 2.9599576537459184e-05, 'p95_s': 0.00962633200003

### GPU StartUp Time correct Fix

In [4]:
import time, torch, torchvision
t0 = time.perf_counter(); torch.zeros(1, device="cuda"); torch.cuda.synchronize()
t_cuda_init = time.perf_counter() - t0
t0 = time.perf_counter()
m = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval().to("cuda")
t_load_to_gpu = time.perf_counter() - t0
x = torch.randn(1, 3, 224, 224, device="cuda")
t0 = time.perf_counter()
with torch.inference_mode(): m(x)
torch.cuda.synchronize()
t_first = time.perf_counter() - t0
print(round(t_cuda_init,3), round(t_load_to_gpu,3), round(t_first,4))

0.001 0.101 0.0059


## TPU Model Testing

In [2]:
import copy, time, statistics, csv, torch, torch_tpu, torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
dev = torch.device("tpu")
print("PyTorch", torch.__version__, "| device:", dev)

def sync(out):
    out.sum().item()          # forces the TPU to finish, brings a value back to host

# ---------- model load, then first forward (includes XLA compile) ----------
t0 = time.perf_counter()
base = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval()
t_load = time.perf_counter() - t0
base = base.to(dev)
x1 = torch.randn(1, 3, 224, 224, device=dev)
t1 = time.perf_counter()
with torch.inference_mode():
    sync(base(x1))
t_first = time.perf_counter() - t1
print("load_s:", round(t_load, 3), "| first_forward_s (incl. compile):", round(t_first, 4))

# ---------- speed ----------
compile_times = {}
def bench(m, dtype_name, dtype, bs, reps):
    x = torch.randn(bs, 3, 224, 224, device=dev, dtype=dtype)
    with torch.inference_mode():
        t0 = time.perf_counter(); sync(m(x))          # first call at this shape = compile
        compile_times[(dtype_name, bs)] = time.perf_counter() - t0
        for _ in range(2):
            sync(m(x))
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); out = m(x); sync(out); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "tpu", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_img_s": bs / med,
            "compile_s": compile_times[(dtype_name, bs)], "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16), ("fp16", torch.float16)]:
    m = copy.deepcopy(base).to(dt).to(dev)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "tpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:120]}"})
        print(rows[-1])

# ---------- accuracy: same 1000-image subset (seed 0) ----------
tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                         transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
ds = datasets.ImageFolder("imagenette2-320/val", transform=tf)
g = torch.Generator().manual_seed(0)
idx = torch.randperm(len(ds), generator=g)[:1000].tolist()
loader = DataLoader(Subset(ds, idx), batch_size=32, num_workers=0)
IMAGENET_IDX = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])

def accuracy(m, dtype):
    correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            logits = m(x.to(dev, dtype)).cpu()[:, IMAGENET_IDX]
            correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return correct / total

for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16), ("fp16", torch.float16)]:
    try:
        m = copy.deepcopy(base).to(dt).to(dev)
        acc = accuracy(m, dt)
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy", "reps": 1000,
                     "accuracy": acc, "status": "ok"})
        print(name, "accuracy:", acc)
    except Exception as e:
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy",
                     "status": f"failed: {type(e).__name__}: {str(e)[:120]}"})

with open("mobilenetv2_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved mobilenetv2_tpu.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


PyTorch 2.14.1+cpu | device: tpu
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 114MB/s]


load_s: 0.302 | first_forward_s (incl. compile): 4.0514
{'device': 'tpu', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.012710954999988644, 'std_s': 0.0017566520324930192, 'p95_s': 0.01343403999999282, 'p99_s': 0.013715609999962908, 'throughput_img_s': 78.67229488271286, 'compile_s': 0.007909219999987727, 'status': 'ok'}
{'device': 'tpu', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.01238897499996483, 'std_s': 0.00022946428889787467, 'p95_s': 0.012658719999990353, 'p99_s': 0.012658719999990353, 'throughput_img_s': 161.43385550505008, 'compile_s': 6.14729862799993, 'status': 'ok'}
{'device': 'tpu', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.011141844999997375, 'std_s': 0.00045566208660331785, 'p95_s': 0.011749450000024808, 'p99_s': 0.011749450000024808, 'throughput_img_s': 359.00696877410724, 'compile_s': 6.815679388000035, 'status': 'ok'}
{'device': 'tpu', 'precision': 'fp32', 'batch': 8, 'reps': 10, 'median_s': 0.01478922949996786, 'std_s': 